# 03 — Change Detection (Phase 3)

**Project:** DGCA Agentic AI Regulatory Intelligence & Monitoring System
**Phase:** 3 of 8 — Persistent Storage & Change Detection

## 1. Project Overview

Phase 2 (`02_DGCA_Website_Monitor.ipynb`) scrapes DGCA's "What's New" section
and saves a timestamped JSON snapshot to `data/raw/`. On its own that
snapshot is stateless -- every run looks identical whether nothing changed or
DGCA published ten new items. This notebook adds the missing piece: **a
PostgreSQL-backed memory of what has already been seen**, so the pipeline can
tell the difference between a publication that is genuinely new, one that's
already been handled, one that DGCA has revised, and one that failed to
process -- and, critically, so it never sends the same email notification
twice for the same unchanged publication.

**Note on notebook numbering:** the project brief's suggested file list named
this slot `03_Document_Processing.ipynb`, but the brief's own numbered phase
list defines **Phase 3 as Change Detection** and **Phase 4 as Document
Processing** (PDF download/OCR) -- two genuinely different concerns that each
need independent testing per the project's incremental build discipline.
This notebook is `03_Change_Detection.ipynb`; PDF download/text extraction
will be `04_Document_Processing.ipynb` when that phase starts.

```
Phase 2 snapshot (data/raw/whats_new_*.json)
        |
        v
ingest_batch()  -- per publication, compared against PostgreSQL by publication_id:
        |
        +-- not in DB               -> NEW
        +-- in DB, hash unchanged   -> UNCHANGED (already processed, no-op)
        +-- in DB, hash changed     -> UPDATED (DGCA revised it -- re-notify)
        +-- exception while ingesting -> FAILED (logged, batch continues)
        v
PostgreSQL publications via `DATABASE_URL`
        |
        v
get_pending_notifications()  -- only NEW/UPDATED rows not yet emailed
        |                        (this is what prevents duplicate emails)
        v
mark_email_sent() / mark_processed() / mark_for_resend() (explicit retry)
```

**Scope of this phase:** the storage + classification logic only, tested
against real Phase 2 output already on disk. Wiring this live to the Phase 1
email pipeline and Phase 2's scraper is a later integration step (Phase 7's
agentic orchestration, or an earlier lightweight glue script) -- deliberately
not done here.


## 2. Imports

In [ ]:
import hashlib
import json
import logging
from psycopg import Connection
from psycopg.rows import DictRow
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from pathlib import Path
from typing import Any, Dict, List, Optional

print("Imports OK")


## 3. Configuration

No secrets here (a local database file). `PROJECT_ROOT` is resolved the same
way as the other notebooks so this runs correctly regardless of Jupyter's
working directory.


In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
LOG_DIR = PROJECT_ROOT / "logs"

LOG_DIR.mkdir(parents=True, exist_ok=True)


print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"DB_PATH      : PostgreSQL (DATABASE_URL)")

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / "config" / ".env")


## 4. Logging

In [ ]:
LOG_FILE = LOG_DIR / f"change_detection_{datetime.now(timezone.utc).strftime('%Y%m%d')}.log"

logger = logging.getLogger("dgca_change_detection")
logger.setLevel(logging.INFO)
logger.handlers.clear()

_formatter = logging.Formatter("[%(asctime)s] [%(levelname)s] %(message)s", datefmt="%Y-%m-%d %H:%M:%S")

_console_handler = logging.StreamHandler()
_console_handler.setFormatter(_formatter)
logger.addHandler(_console_handler)

_file_handler = logging.FileHandler(LOG_FILE, encoding="utf-8")
_file_handler.setFormatter(_formatter)
logger.addHandler(_file_handler)

logger.propagate = False

logger.info("Change detection logger initialized.")
logger.info(f"Log file: {LOG_FILE}")


## 5. Database Schema

Matches the project brief's schema exactly. `publication_id` reuses Phase
2's `source_id` (category + decoded attachment id) as the natural primary
key -- it's already designed to be stable across scrapes of the same
publication.


In [ ]:
from pipeline.storage import get_connection, init_db

init_db()


## 6. Content Hashing

`content_hash` is a SHA-256 over the fields that matter for "did DGCA change
this publication" -- title, category, published date, and PDF URL. A
different `document_hash` (the actual PDF's bytes, hashed) is reserved for
Phase 4, once PDFs are actually downloaded; it stays `NULL` until then.


In [ ]:
def compute_content_hash(pub: Dict[str, Any]) -> str:
    parts = [
        str(pub.get("title", "")),
        str(pub.get("category", "")),
        str(pub.get("published_date", "")),
        str(pub.get("pdf_url", "")),
    ]
    joined = "|".join(parts)
    return hashlib.sha256(joined.encode("utf-8")).hexdigest()


print("compute_content_hash() ready.")


## 7. Ingestion Logic — classifying and upserting a single record

`ingest_publication()` never raises: any failure (missing required field,
unexpected type, DB error) is caught, logged, and reported as an
`IngestOutcome.FAILED` for that one record -- the existing DB row (if any)
is left untouched rather than risking corrupting good data with a partial
write.


In [ ]:
class IngestOutcome(str, Enum):
    NEW = "new"
    UPDATED = "updated"
    UNCHANGED = "unchanged"
    FAILED = "failed"


REQUIRED_FIELDS = ("source_id", "title", "category")


def ingest_publication(conn: Connection, pub: Dict[str, Any]) -> IngestOutcome:
    try:
        missing = [f for f in REQUIRED_FIELDS if not pub.get(f)]
        if missing:
            raise ValueError(f"missing required field(s): {missing}")

        publication_id = pub["source_id"]
        content_hash = compute_content_hash(pub)
        now = datetime.now(timezone.utc).isoformat()

        existing = conn.execute(
            "SELECT content_hash, email_sent FROM publications WHERE publication_id = %s",
            (publication_id,),
        ).fetchone()

        if existing is None:
            conn.execute(
                """
                INSERT INTO publications
                    (publication_id, title, category, published_date, publication_url,
                     pdf_url, content_hash, document_hash, first_seen_at, processed_at,
                     email_sent, email_sent_at, status)
                VALUES (%s, %s, %s, %s, %s, %s, %s, NULL, %s, NULL, 0, NULL, 'new')
                """,
                (
                    publication_id, pub["title"], pub["category"], pub.get("published_date"),
                    pub.get("publication_url"), pub.get("pdf_url"), content_hash, now,
                ),
            )
            conn.commit()
            logger.info(f"[NEW] {publication_id} -- {pub['title'][:60]}")
            return IngestOutcome.NEW

        if existing["content_hash"] == content_hash:
            logger.info(f"[UNCHANGED] {publication_id}")
            return IngestOutcome.UNCHANGED

        # Content differs from what we stored -- DGCA revised this publication.
        # Reset the notification state so it gets picked up as pending again.
        conn.execute(
            """
            UPDATE publications
            SET title = %s, category = %s, published_date = %s, publication_url = %s,
                pdf_url = %s, content_hash = %s, status = 'updated',
                email_sent = 0, email_sent_at = NULL, processed_at = NULL
            WHERE publication_id = %s
            """,
            (
                pub["title"], pub["category"], pub.get("published_date"),
                pub.get("publication_url"), pub.get("pdf_url"), content_hash, publication_id,
            ),
        )
        conn.commit()
        logger.info(f"[UPDATED] {publication_id} -- content changed since last seen")
        return IngestOutcome.UPDATED

    except Exception as exc:
        logger.error(f"[FAILED] Could not ingest record ({exc.__class__.__name__}): {exc}")
        return IngestOutcome.FAILED


print("ingest_publication() ready.")


## 8. Batch Ingestion

Runs `ingest_publication()` over a whole scrape, tallying outcomes. One bad
record (Section 12's failure test) never aborts the batch -- every other
record still gets processed.


In [ ]:
@dataclass
class IngestSummary:
    new: int = 0
    updated: int = 0
    unchanged: int = 0
    failed: int = 0
    failed_ids: List[str] = field(default_factory=list)
    total: int = 0


def ingest_batch(conn: Connection, publications: List[Dict[str, Any]]) -> IngestSummary:
    summary = IngestSummary(total=len(publications))
    for pub in publications:
        outcome = ingest_publication(conn, pub)
        if outcome == IngestOutcome.NEW:
            summary.new += 1
        elif outcome == IngestOutcome.UPDATED:
            summary.updated += 1
        elif outcome == IngestOutcome.UNCHANGED:
            summary.unchanged += 1
        else:
            summary.failed += 1
            summary.failed_ids.append(pub.get("source_id", "<unknown>"))

    logger.info(
        f"Batch ingest complete: {summary.total} total -- "
        f"new={summary.new}, updated={summary.updated}, "
        f"unchanged={summary.unchanged}, failed={summary.failed}"
    )
    return summary


print("ingest_batch() ready.")


## 9. Query Helpers — dedup-safe notification queue

`get_pending_notifications()` is the function a future email-integration step
would call: it returns only publications that are `new`/`updated` **and**
have not already had an email sent for their current content -- so re-running
ingestion on unchanged data can never produce a duplicate notification.
`mark_for_resend()` is the explicit escape hatch the brief calls for
("never duplicate emails **unless explicitly configured for retry**").


In [ ]:
def get_pending_notifications(conn: Connection) -> List[DictRow]:
    return conn.execute(
        "SELECT * FROM publications WHERE status IN ('new', 'updated') AND email_sent = 0 "
        "ORDER BY first_seen_at"
    ).fetchall()


def mark_email_sent(conn: Connection, publication_id: str) -> None:
    now = datetime.now(timezone.utc).isoformat()
    conn.execute(
        "UPDATE publications SET email_sent = 1, email_sent_at = %s, status = 'processed', "
        "processed_at = %s WHERE publication_id = %s",
        (now, now, publication_id),
    )
    conn.commit()
    logger.info(f"[email-sent] {publication_id}")


def mark_processed(conn: Connection, publication_id: str) -> None:
    now = datetime.now(timezone.utc).isoformat()
    conn.execute(
        "UPDATE publications SET status = 'processed', processed_at = %s WHERE publication_id = %s",
        (now, publication_id),
    )
    conn.commit()


def mark_for_resend(conn: Connection, publication_id: str) -> None:
    """Explicit, operator-triggered retry: clears email_sent without touching
    content_hash, so the next notification pass will re-send this publication
    even though its content hasn't changed."""
    conn.execute(
        "UPDATE publications SET email_sent = 0, email_sent_at = NULL WHERE publication_id = %s",
        (publication_id,),
    )
    conn.commit()
    logger.info(f"[resend-requested] {publication_id}")


def get_all(conn: Connection) -> List[DictRow]:
    return conn.execute("SELECT * FROM publications ORDER BY first_seen_at").fetchall()


print("Query helpers ready.")


## 10. Load Phase 2 Scraped Data

Loads the most recent snapshot Phase 2's scraper already produced -- this
notebook doesn't re-scrape DGCA itself, keeping it independently testable.


In [ ]:
snapshot_files = sorted(RAW_DATA_DIR.glob("whats_new_*.json"))
if not snapshot_files:
    raise FileNotFoundError(
        f"No Phase 2 snapshots found in {RAW_DATA_DIR}. Run 02_DGCA_Website_Monitor.ipynb first."
    )

latest_snapshot_path = snapshot_files[-1]
snapshot_data = json.loads(latest_snapshot_path.read_text(encoding="utf-8"))
scraped_publications = snapshot_data["publications"]

print(f"Loaded snapshot: {latest_snapshot_path.name}")
print(f"Publications in snapshot: {len(scraped_publications)}")
print(f"Category counts: {snapshot_data.get('category_counts')}")


## 11. Test 1 — Fresh Ingest (all records classified `NEW`)

A clean database, ingesting the scraped batch for the first time. Every
record should come back `NEW`.


In [ ]:
# Keep the demonstration reproducible with a temporary table in this direct connection.
conn = get_connection()
conn.execute("CREATE TEMP TABLE publications (LIKE public.publications INCLUDING ALL)")

summary_1 = ingest_batch(conn, scraped_publications)
print(summary_1)
assert summary_1.new == len(scraped_publications), "Expected every record to be NEW on a clean database."
assert summary_1.updated == 0 and summary_1.unchanged == 0 and summary_1.failed == 0
print("Test 1 passed: fresh ingest classified every record as NEW.")


## 12. Test 2 — Re-ingest the Same Batch (duplicate-email prevention)

First, simulate the email pipeline having actually sent notifications for
every pending record (via `mark_email_sent()`, the same call a real
Phase 1 integration would make). Then re-ingest the **identical** snapshot
data and confirm: nothing comes back `NEW`, and — the real point — the
pending-notifications queue is empty, meaning no duplicate email would ever
be triggered for unchanged publications.


In [ ]:
pending_before = get_pending_notifications(conn)
print(f"Pending notifications after Test 1: {len(pending_before)}")
assert len(pending_before) == len(scraped_publications)

for row in pending_before:
    mark_email_sent(conn, row["publication_id"])

pending_after_send = get_pending_notifications(conn)
assert len(pending_after_send) == 0, "Expected zero pending notifications right after marking all as sent."
print(f"Pending notifications after marking all sent: {len(pending_after_send)}")

# Re-ingest the exact same data DGCA "still has" -- nothing changed.
summary_2 = ingest_batch(conn, scraped_publications)
print(summary_2)
assert summary_2.new == 0 and summary_2.updated == 0 and summary_2.failed == 0
assert summary_2.unchanged == len(scraped_publications)

pending_duplicate_check = get_pending_notifications(conn)
assert len(pending_duplicate_check) == 0, "Duplicate-prevention failed: unchanged records should never re-enter the pending queue."
print("Test 2 passed: re-ingesting unchanged data produced zero new pending notifications (no duplicate emails).")


## 13. Test 3 — Simulated DGCA Revision (`UPDATED` detection)

Mutates one already-processed record's `pdf_url` (as if DGCA replaced the
attachment on an existing publication) and re-ingests. This must be detected
as `UPDATED` — distinct from both `NEW` and `UNCHANGED` — and must
re-enter the pending-notifications queue, since a genuine content change
*should* trigger a fresh notification (this is not a duplicate).


In [ ]:
revised_publications = [dict(p) for p in scraped_publications]
target = revised_publications[0]
original_pdf_url = target["pdf_url"]
target["pdf_url"] = original_pdf_url + "?revision=2"
print(f"Simulating a DGCA revision for: {target['title'][:70]}")

summary_3 = ingest_batch(conn, revised_publications)
print(summary_3)
assert summary_3.updated == 1, f"Expected exactly 1 UPDATED record, got {summary_3.updated}"
assert summary_3.new == 0 and summary_3.failed == 0
assert summary_3.unchanged == len(revised_publications) - 1

pending_after_update = get_pending_notifications(conn)
pending_ids = {row["publication_id"] for row in pending_after_update}
assert target["source_id"] in pending_ids, "The revised publication should be back in the pending-notification queue."
print(f"Test 3 passed: revision correctly classified as UPDATED and re-queued for notification ({len(pending_after_update)} pending).")


## 14. Test 4 — Simulated Bad Record (resilience)

A batch with one malformed record (missing `title`) mixed in with otherwise
valid ones. The bad record must be reported as `FAILED` without raising, and
— critically — every other record in the same batch must still be processed
normally.


In [ ]:
# Use revised_publications (Test 3's data) as the "currently valid" baseline --
# the DB now reflects that revision, so re-ingesting the pre-revision data
# here would itself look like another change. Using the same baseline as
# what's actually stored keeps this test isolated to the one broken record.
mixed_batch = [dict(p) for p in revised_publications[:3]]
broken_record = dict(mixed_batch[0])
broken_record["source_id"] = "regulation-deliberately-broken-test-record"
del broken_record["title"]  # missing required field
mixed_batch.append(broken_record)

summary_4 = ingest_batch(conn, mixed_batch)
print(summary_4)
assert summary_4.failed == 1, f"Expected exactly 1 FAILED record, got {summary_4.failed}"
assert "regulation-deliberately-broken-test-record" in summary_4.failed_ids
assert summary_4.unchanged == 3, "The 3 valid records in the same batch should still have been processed (as UNCHANGED)."
print("Test 4 passed: one bad record failed gracefully without affecting the other 3 records in the same batch.")


## 15. Final Database State

A full dump of the table after all four tests, for a sanity check of exactly
what's persisted.


In [ ]:
all_rows = get_all(conn)
print(f"Total rows in database: {len(all_rows)}")
print()
for row in all_rows[:5]:
    print(f"- [{row['status']:<9}] {row['publication_id']:<40} email_sent={bool(row['email_sent'])}  {row['title'][:60]}")
if len(all_rows) > 5:
    print(f"... and {len(all_rows) - 5} more.")


## 16. Results

In [ ]:
print("=" * 70)
print("CHANGE DETECTION TEST RESULTS")
print("=" * 70)
print(f"Test 1 (fresh ingest)         : PASSED -- {summary_1.new} new")
print(f"Test 2 (duplicate prevention) : PASSED -- {summary_2.unchanged} unchanged, 0 duplicate notifications")
print(f"Test 3 (update detection)     : PASSED -- {summary_3.updated} updated, correctly re-queued")
print(f"Test 4 (failure resilience)   : PASSED -- {summary_4.failed} failed record isolated, others unaffected")
print(f"Database location             : PostgreSQL (DATABASE_URL)")
print(f"Total rows persisted          : {len(all_rows)}")
print("=" * 70)

conn.close()

# Keep the real scrape available to the document-processing phase.
from pipeline.storage import ingest_batch as persist_publications
with get_connection() as persistent_conn:
    persist_publications(persistent_conn, scraped_publications)


## 17. Next Steps

**Phase 3 status: PostgreSQL change-detection storage built and tested** against
real Phase 2 scrape data, with all four required scenarios (new, duplicate
prevention, update detection, failure resilience) verified.

### Known integration work, deliberately not done here

- This notebook loads a Phase 2 snapshot **file**; nothing yet calls Phase
  2's scraper and Phase 3's `ingest_batch()` back-to-back in one run. That
  glue (and, later, calling Phase 1's `send_dgca_notification()` for each row
  `get_pending_notifications()` returns) is either a lightweight orchestration
  script or Phase 7's agentic workflow -- worth deciding explicitly rather
  than assuming.
- `document_hash` stays `NULL` until Phase 4 downloads and hashes actual PDF
  bytes.
- No pruning/archival policy for old rows yet -- not needed until the table
  has real history to manage.

### Not started yet (by design)

- Phase 4: PDF download & text extraction (`04_Document_Processing.ipynb`)
- Phase 5: Old-vs-new AI comparison
- Phase 6: AI summary integration into the scraped-data flow
- Phase 7: Agentic workflow orchestration
- Phase 8: Streamlit dashboard

Waiting for confirmation before starting Phase 4.
